# Amazon ESCI 日本語検索 — XGBoost LTR v3

## 目的と結論

v2の11 lexical特徴へ、全語一致・raw完全一致・色・数字/型番・否定/反対条件・query typeを追加し、31特徴のXGBoost LTRを実装した。文書依存featureはOpenSearch LTR feature logで計算し、query側の決定的な前処理だけをPythonと検索サービスで共有する。

train内validationではv2特徴の公式nDCG **0.84487** に対してv3は **0.85147**。差分 **+0.00659** のpaired bootstrap 95% CIは **[+0.00321, +0.00999]** で、モデル選択に使っていないvalidation上で改善した。corpus top-100再順位のnDCG@10（未判定=0）はv2 **0.40238** からv3 **0.42467**。候補集合は同じなのでRecall@100は変化しない。

## v3の特徴設計

|群|追加特徴|狙い|
|---|---|---|
|coverage/exact|`title_and`, `brand_and`, `bullet_and`, `description_and`, `title_raw_exact`, `brand_raw_exact`, `color_bm25`|全条件充足、ブランド・完全商品名|
|code/constraint|`code_title`, `code_brand`, `numeric_title`, `constraint_title_and`|A4、iqos3、型番、否定語を除いた条件|
|negation|`negation_title`, `negation_bullet`, `contrary_title`, `contrary_bullet`|「なし」と「付き」などを別信号にする|
|query type|`query_token_count`, `has_numeric`, `has_latin`, `has_negation`, `is_short_query`|query種別に応じて木の分岐を変える|

OpenSearch LTR 3.7の欠損分岐制約に合わせ、全featureは非一致時も`1e-6`で明示的にmatchする。query前処理はNFKC・lowercase、数字/英数字、否定marker、反対語、制約部分を抽出する。

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Markdown
sys.path.insert(0, str(Path.cwd()))
from ltr_features import query_params

A = Path("artifacts/ltr_xgboost_v3")
metrics = json.loads((A / "metrics.json").read_text())
tuning = pd.read_csv(A / "validation_tuning.csv")
ablation = pd.read_csv(A / "feature_ablation.csv")
importance = pd.read_csv(A / "feature_importance.csv")
slices = pd.read_csv(A / "validation_query_slices.csv")
diagnostic = pd.read_csv(A / "diagnostic_30.csv")
parity = pd.read_csv(A / "parity_check.csv")
print("loaded", A.resolve())

loaded /Users/masamichiimaseki/Documents/opensearch-trial/amazon/artifacts/ltr_xgboost_v3


## 1. query前処理の例

In [2]:
display(pd.DataFrame([query_params(q) for q in [
    "PP袋 A4 テープなし", "iqos3 ケース", "フラミンゴ"
]], index=["PP袋 A4 テープなし", "iqos3 ケース", "フラミンゴ"]).T)

,PP袋 A4 テープなし,iqos3 ケース,フラミンゴ
keywords,pp袋 a4 テープなし,iqos3 ケース,フラミンゴ
normalized_query,pp袋 a4 テープなし,iqos3 ケース,フラミンゴ
code_tokens,a4 4,iqos3 3,zzzzltrnomatchzzzz
numeric_tokens,4,3,zzzzltrnomatchzzzz
negation_terms,テープなし,zzzzltrnomatchzzzz,zzzzltrnomatchzzzz
contrary_terms,テープ付 テープ付き,zzzzltrnomatchzzzz,zzzzltrnomatchzzzz
constraint_keywords,pp袋 a4 テープ,iqos3 ケース,フラミンゴ
query_token_count,4.000001,2.000001,1.000001
has_numeric,1.0,1.0,0.000001
has_latin,1.0,1.0,0.000001


## 2. train内validationでのモデル選択

既存Notebookと同じ `int(SHA1(query_id),16) % 5 == 0` の1,440 queryをvalidationに固定。5,844 queryだけでfitし、v2/v3それぞれ`rank:pairwise`/`rank:ndcg`、depth 3/5を比較した。testは選択に使用しない。

In [3]:
display(tuning.sort_values(["feature_version", "validation_official_nDCG"], ascending=[True, False]).round(6))
p = metrics["validation_v3_vs_v2"]
display(Markdown(f"**v3−v2: {p['delta']:+.6f}**, 95% CI [{p['ci95_low']:+.6f}, {p['ci95_high']:+.6f}], {p['wins']}勝/{p['ties']}同/{p['losses']}敗。"))

,feature_version,config,objective,max_depth,trees,validation_official_nDCG
2,v2_features,ndcg_d3,rank:ndcg,3,55,0.844872
1,v2_features,pairwise_d5,rank:pairwise,5,69,0.844665
3,v2_features,ndcg_d5,rank:ndcg,5,49,0.844395
0,v2_features,pairwise_d3,rank:pairwise,3,135,0.843754
7,v3_features,ndcg_d5,rank:ndcg,5,150,0.851465
6,v3_features,ndcg_d3,rank:ndcg,3,172,0.849604
4,v3_features,pairwise_d3,rank:pairwise,3,178,0.849109
5,v3_features,pairwise_d5,rank:pairwise,5,47,0.848777


**v3−v2: +0.006593**, 95% CI [+0.003214, +0.009986], 674勝/241同/525敗。

## 3. feature ablation

In [4]:
display(ablation.round(6))
display(Markdown("coverage/exactだけの増分は小さくCIがゼロを跨ぐ。一方、型番・否定・constraintまで加えると有意な増分になり、query-type特徴でさらに伸びた。"))

,feature_group,features,trees,validation_official_nDCG,delta_vs_v2,ci95_low,ci95_high
0,v2_11,11,49,0.844395,0.000000,0.000000,0.000000
1,plus_coverage_exact_18,18,159,0.846083,0.001689,-0.001287,0.004749
2,plus_code_negation_26,26,58,0.849825,0.005430,0.002163,0.008786
3,v3_all_31,31,150,0.851465,0.007070,0.003604,0.010679


coverage/exactだけの増分は小さくCIがゼロを跨ぐ。一方、型番・否定・constraintまで加えると有意な増分になり、query-type特徴でさらに伸びた。

## 4. query slice

In [5]:
slice_rows = []
for flag in ["has_numeric", "has_latin", "has_negation", "is_short_query"]:
    for value, group in slices.groupby(flag):
        slice_rows.append({"slice": flag, "value": bool(value), "queries": len(group), "mean_delta_v3_v2": group.delta_v3_v2.mean()})
display(pd.DataFrame(slice_rows).round(6))

,slice,value,queries,mean_delta_v3_v2
0,has_numeric,False,1227,0.001295
1,has_numeric,True,213,0.040339
2,has_latin,False,1076,0.002678
3,has_latin,True,364,0.020056
4,has_negation,False,1263,0.006935
5,has_negation,True,177,0.008037
6,is_short_query,False,1306,0.007237
7,is_short_query,True,134,0.005448


## 5. held-out reportingとcorpus再順位

testはv2開発時に既に参照済みなので、ここでは新しい未観測validationとしては扱わず、再現確認用のreporting setとする。主要な採否根拠は上のtrain内validation。

In [6]:
summary = pd.DataFrame([
    {"metric": "Task1 candidate official nDCG", "v2": metrics['test_v2_official_nDCG'], "v3": metrics['test_v3_official_nDCG']},
    {"metric": "corpus nDCG@10 unjudged=0", "v2": metrics['end_to_end_v2_nDCG_at_10_unjudged0'], "v3": metrics['end_to_end_v3_nDCG_at_10_unjudged0']},
    {"metric": "corpus nDCG@100 unjudged=0", "v2": metrics['end_to_end_v2_nDCG_at_100_unjudged0'], "v3": metrics['end_to_end_v3_nDCG_at_100_unjudged0']},
    {"metric": "Recall@100 judged positive", "v2": metrics['end_to_end_v2_recall_at_100_judged_positive'], "v3": metrics['end_to_end_v3_recall_at_100_judged_positive']},
])
summary["delta"] = summary.v3 - summary.v2
display(summary.round(6))

,metric,v2,v3,delta
0,Task1 candidate official nDCG,0.849470,0.855483,0.006013
1,corpus nDCG@10 unjudged=0,0.402380,0.424668,0.022288
2,corpus nDCG@100 unjudged=0,0.475101,0.487602,0.012502
3,Recall@100 judged positive,0.485190,0.485190,0.000000


## 6. 当初の30問題query

small-version学習対象外14件は、v2平均0.59476からv3 0.64432（+0.04956、6勝/3同/5敗）。ただし14件だけの95% CIはゼロを跨ぐため、方向確認用とする。

In [7]:
cols = ["query_id", "query", "query_type", "training_status", "v2", "v3", "delta_v3_v2"]
display(diagnostic.sort_values("delta_v3_v2", ascending=False)[cols].round(6))

,query_id,query,query_type,training_status,v2,v3,delta_v3_v2
25,127228,文豪ストレイドッグス ねんどろいど,作品＋商品タイプ,small-version学習集合内,0.540201,0.993996,0.453795
2,41495,flower,英語の広いカテゴリ,学習対象外,0.129875,0.563788,0.433913
6,82972,pp袋 a4 テープなし,属性・否定条件,small-version学習集合内,0.078398,0.408073,0.329675
12,120267,スズキ スマートキーケース,ブランド＋適合,small-version学習集合内,0.393758,0.652864,0.259106
24,126907,思い 思われ ふり ふられ,作品タイトル,学習対象外,0.654932,0.889954,0.235022
20,124760,レインボーシックス シージ,作品名,学習対象外,0.641046,0.751092,0.110046
9,114017,wrc 8: fia world rally championship,作品名＋型番,学習対象外,0.796897,0.895351,0.098454
7,92648,shkkn kelly,typo/未知ブランド,学習対象外,0.914857,1.000000,0.085143
3,51965,hugo boss shoes,ブランド＋カテゴリ,学習対象外,0.796761,0.866948,0.070187
17,123338,ペットマーキング防止スプレー,用途・課題解決,small-version学習集合内,0.002201,0.065009,0.062808


## 7. 特徴重要度とOpenSearch parity

In [8]:
display(importance.head(20).round(5))
display(pd.DataFrame([{
    "model": "esci_jp_xgb_v3", "featureset": "esci_jp_features_v3",
    "max_abs_score_diff": metrics['parity_max_abs_diff'],
    "rank_equal": metrics['parity_rank_equal'], "checked_rows": len(parity),
}]))

,feature,gain
0,title_and,0.34048
1,lexical_v2,0.14635
2,constraint_title_and,0.06842
3,brand_phrase,0.04734
4,title_ngram,0.04207
5,has_numeric,0.03799
6,brand_and,0.03349
7,title_reading,0.03254
8,title_raw_exact,0.02731
9,code_title,0.01763


,model,featureset,max_abs_score_diff,rank_equal,checked_rows
0,esci_jp_xgb_v3,esci_jp_features_v3,0.000002,True,96


## 配備方法と判断

OpenSearchへfeature set `esci_jp_features_v3`、model `esci_jp_xgb_v3`を登録済み。検索時には必ず`query_params(query)`相当のparamsを`sltr`へ渡し、`window_size=100`, `query_weight=0`, `rescore_query_weight=1`で再順位する。Python/OpenSearchの順位は完全一致した。

v3は採用可能。ただし`iqos3 ケース`、`レーザー墨出し`、`ワイヤレス充電器`など個別悪化は残る。またRecall@100は不変であり、候補に入らない意味的関連商品は救えない。次段階はembeddingによるsemantic候補をlexical候補とunion/RRFし、そのsemantic scoreをv3 LTRへ加える。